In [2]:
from pysat.formula import CNF, IDPool
from pysat.solvers import Solver, Minisat22
from pysat.card import CardEnc, EncType

In [3]:
# generate CNF encoding given length and specific lag
def generate_encoding_lag(r, lag, cnf, pool, varmap, pmap_lags, seqs):
    # add variable to map
    def seq_id(seq, i):
        key = f'{seq}_{i}'
        if key not in varmap:
            varmap[key] = pool.id(key)
        return varmap[key]

    # create auxiliary variable for x * y
    def add_reified_xnor(p, x, y):
        cnf.append([-p, -x,  y])
        cnf.append([-p,  x, -y])
        cnf.append([ p, -x, -y])
        cnf.append([ p,  x,  y])

    # for each sequence, add clause for new combination
    pmap = {}
    p_lits = []
    for seq in seqs:
        for i in range(r - lag):
            j = i + lag
            seq_i = seq_id(seq, i)
            seq_j = seq_id(seq, j)
            p_key = f'p_{seq}_{i}_{j}'
            p_id = pool.id(p_key)
            pmap[p_key] = p_id
            p_lits.append(p_id)
            add_reified_xnor(p_id, seq_i, seq_j)

    # verify sum is 0
    eq = CardEnc.equals(lits=p_lits, bound=2 * (r - lag), encoding=EncType.totalizer, vpool=pool)
    cnf.extend(eq.clauses)
    pmap_lags[lag] = pmap

In [4]:
# generate CNF encoding given length
def generate_encoding(r):
    cnf = CNF()
    pool = IDPool()
    varmap = {}
    pmap_lags = {}
    seqs = ['X', 'Y', 'Z', 'W']

    for lag in range(1, r):
        generate_encoding_lag(r, lag, cnf, pool, varmap, pmap_lags, seqs)
        
    return cnf, varmap, pmap_lags, pool

In [5]:
cnf, varmap, pmap_lags, pool = generate_encoding(7)
print(varmap)
print(cnf.clauses)

{'X_0': 1, 'X_1': 2, 'X_2': 4, 'X_3': 6, 'X_4': 8, 'X_5': 10, 'X_6': 12, 'Y_0': 14, 'Y_1': 15, 'Y_2': 17, 'Y_3': 19, 'Y_4': 21, 'Y_5': 23, 'Y_6': 25, 'Z_0': 27, 'Z_1': 28, 'Z_2': 30, 'Z_3': 32, 'Z_4': 34, 'Z_5': 36, 'Z_6': 38, 'W_0': 40, 'W_1': 41, 'W_2': 43, 'W_3': 45, 'W_4': 47, 'W_5': 49, 'W_6': 51}
[[-3, -1, 2], [-3, 1, -2], [3, -1, -2], [3, 1, 2], [-5, -2, 4], [-5, 2, -4], [5, -2, -4], [5, 2, 4], [-7, -4, 6], [-7, 4, -6], [7, -4, -6], [7, 4, 6], [-9, -6, 8], [-9, 6, -8], [9, -6, -8], [9, 6, 8], [-11, -8, 10], [-11, 8, -10], [11, -8, -10], [11, 8, 10], [-13, -10, 12], [-13, 10, -12], [13, -10, -12], [13, 10, 12], [-16, -14, 15], [-16, 14, -15], [16, -14, -15], [16, 14, 15], [-18, -15, 17], [-18, 15, -17], [18, -15, -17], [18, 15, 17], [-20, -17, 19], [-20, 17, -19], [20, -17, -19], [20, 17, 19], [-22, -19, 21], [-22, 19, -21], [22, -19, -21], [22, 19, 21], [-24, -21, 23], [-24, 21, -23], [24, -21, -23], [24, 21, 23], [-26, -23, 25], [-26, 23, -25], [26, -23, -25], [26, 23, 25], [-2

In [30]:
def make_assumptions(varmap, assign):
    lits = []
    for (seq, i), val in assign.items():
        v = varmap[f'{seq}_{i}']
        lits.append(v if val == 1 else -v)
    return lits

def verify_assignment(assignment, varmap, cnf):
    assign = {}
    seqs = ['X', 'Y', 'Z', 'W']

    seq_num = 0
    for seq in seqs:
        for i in range(len(assignment[seq_num])):
            assign[(seq, i)] = assignment[seq_num][i]
        seq_num += 1

    with Minisat22(bootstrap_with=cnf) as solver:
        print('formula is', f'{"s" if solver.solve(assumptions=make_assumptions(varmap, assign)) else "uns"}atisfiable')

In [26]:
from sequence_generator import generate_all_turyns, generate_invalid_turyn

r = 3
turyn_seqs = generate_invalid_turyn(r)
turyn_seqs = [(1, -1, 1), (1, -1, -1), (1, 1, 1), (1, 1, -1)]
cnf, varmap, pmap_lags, pool = generate_encoding(r)
verify_assignment(turyn_seqs, varmap, cnf)

formula is satisfiable
